# Commodity News Scraper (Dual Source)
Fetches commodity news from both:
1. **Investing.com** - Direct web scraping for US Wheat and Crude Oil
2. **SERP API** - Commodity shock news from major financial news sources

SERP API Sources:
- Reuters (reuters.com)
- Bloomberg (bloomberg.com)
- Wall Street Journal (wsj.com)
- Financial Times (ft.com)

Standard columns: commodity, title, date, source, description, url


In [ ]:
# Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'cloudscraper', 'beautifulsoup4', 'lxml', 'google-search-results', 
                       'python-dotenv', 'pandas', 'tqdm', 'ipywidgets', 'jupyter'])


In [ ]:
import os
import sys
import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from dotenv import load_dotenv
from serpapi import GoogleSearch
import time, random, logging

# Load environment variables from .env
load_dotenv()
logging.basicConfig(level=logging.WARNING)

# Force use of standard tqdm (not notebook version) to avoid ipywidgets issues
from tqdm import tqdm

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
print('Ready.')


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

# INVESTING.COM Configuration
INVESTING_CONFIGS = [
    {
        'name': 'wheat',
        'base_url': 'https://www.investing.com/commodities/us-wheat-news',
        'last_page': 46,
    }
]

# SERP API Configuration
SERP_API_KEY = os.getenv('SERP_API')
if not SERP_API_KEY:
    raise ValueError("SERP_API key not found in .env file")

SERP_NEWS_SOURCES = [  
    "bloomberg.com",
    "reuters.com",                
    "cnbc.com",
    "investing.com"
]

SERP_COMMODITIES = [
    {
        'name': 'wheat', 
        'query': 'wheat AND ("prices" OR "supply" OR "exports" OR "futures" OR "market") AND ("war" OR "invasion" OR "tariff" OR "trade war" OR "sanctions" OR "covid" OR "pandemic" OR "drought" OR "crisis" OR "shock")'
    },
]

# Output file (single combined output for both sources)
OUT_CSV = DATA_DIR / 'wheat_news.csv'

# Scraper settings
DELAY_MIN = 2.0
DELAY_MAX = 4.5
CHECKPOINT = 10
MAX_RETRIES = 3

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

# ═══ INVESTING.COM SCRAPER ═══

def make_scraper():
    """Create a cloudscraper session that mimics Chrome on macOS."""
    return cloudscraper.create_scraper(
        browser={'browser': 'chrome', 'platform': 'darwin', 'mobile': False}
    )


def page_url(base_url: str, page: int) -> str:
    """Generate URL for a specific page."""
    if page == 1:
        return base_url
    return f'{base_url}/{page}'


def fetch_page(scraper, url: str, retries: int = MAX_RETRIES):
    """Fetch a URL with retry + exponential back-off. Returns BeautifulSoup or None."""
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.investing.com/',
    }
    for attempt in range(1, retries + 1):
        try:
            r = scraper.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                return BeautifulSoup(r.text, 'lxml')
            elif r.status_code == 404:
                return None
            else:
                print(f'    HTTP {r.status_code} on attempt {attempt}: {url}')
        except Exception as e:
            print(f'    Error attempt {attempt}: {e}')
        if attempt < retries:
            wait = 2 ** attempt + random.uniform(0, 2)
            time.sleep(wait)
    return None


def parse_articles_investing(soup, commodity: str, page: int) -> list[dict]:
    """Extract article records from investing.com news page."""
    records = []
    for art in soup.find_all('article', attrs={'data-test': 'article-item'}):
        # Title + URL
        title_tag = art.find('a', attrs={'data-test': 'article-title-link'})
        title = title_tag.get_text(strip=True) if title_tag else ''
        url = title_tag['href'] if title_tag else ''
        if url and not url.startswith('http'):
            url = 'https://www.investing.com' + url

        # Description / teaser
        desc_tag = art.find('p', attrs={'data-test': 'article-description'})
        description = desc_tag.get_text(strip=True) if desc_tag else ''

        # Source / provider
        src_tag = art.find('a', attrs={'data-test': 'article-provider-link'})
        source = src_tag.get_text(strip=True) if src_tag else 'investing.com'

        # Date
        date_tag = art.find('time', attrs={'data-test': 'article-publish-date'})
        if date_tag:
            date = date_tag.get('datetime', date_tag.get_text(strip=True))
        else:
            date = ''

        if title:  # skip empty / ad slots
            records.append({
                'commodity': commodity,
                'title': title,
                'date': date,
                'source': source,
                'description': description,
                'url': url,
            })
    return records


def already_scraped_pages(csv_path: Path) -> set[int]:
    """Return the set of pages already present in an existing checkpoint CSV."""
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path)
            if 'page_scraped' in df.columns:
                return set(df['page_scraped'].dropna().astype(int).tolist())
        except Exception:
            pass
    return set()


# ═══ SERP API SCRAPER ═══
def fetch_commodity_news_serp(commodity_name: str, query: str, limit_per_year: int = 50) -> list[dict]:
    records = []
    # Loop through specific years to force Google to reveal historical articles
    years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
    
    for source in SERP_NEWS_SOURCES:
        search_query = f'site:{source} {query}'
        
        for year in years:
            year_records = []
            start = 0
            
            while len(year_records) < limit_per_year:
                params = {
                    "q": search_query,
                    "api_key": SERP_API_KEY,
                    "num": 100,
                    "start": start,
                    # Google's custom date range parameter: mm/dd/yyyy
                    "tbs": f"cdr:1,cd_min:01/01/{year},cd_max:12/31/{year}" 
                }
                
                try:
                    search = GoogleSearch(params)
                    results = search.get_dict()
                    
                    if "organic_results" not in results or not results["organic_results"]:
                        break  # No more results for this specific year
                        
                    for result in results["organic_results"]:
                        if len(year_records) >= limit_per_year:
                            break
                        
                        year_records.append({
                            'commodity': commodity_name,
                            'title': result.get('title', ''),
                            'date': result.get('date', ''),
                            'source': source,
                            'description': result.get('snippet', ''),
                            'url': result.get('link', ''),
                        })
                    
                    start += 100 
                    
                except Exception as e:
                    print(f"✗ Error fetching from {source} for year {year}: {str(e)}")
                    break
            
            # Add year_records to main records list AFTER processing all pages for this year
            if year_records:
                records.extend(year_records)
                print(f"✓ Fetched {len(year_records)} articles from {source} ({year})")
    
    return records
# ═══ CSV & DEDUPLICATION ═══

def append_to_csv(records: list[dict], csv_path: Path):
    """Append a list of records to CSV (creates file with header if new)."""
    if not records:
        print(f"  [append_to_csv] No records to append (empty list)")
        return
    
    try:
        df_new = pd.DataFrame(records)
        write_header = not csv_path.exists()
        df_new.to_csv(csv_path, mode='a', header=write_header, index=False)
        print(f"  [append_to_csv] Successfully appended {len(records)} records to {csv_path}")
    except Exception as e:
        print(f"  [append_to_csv] ERROR: {str(e)}")


def deduplicate_records(records: list[dict], existing_urls: set) -> list[dict]:
    """Filter out records that have already been scraped."""
    return [r for r in records if r['url'] not in existing_urls]


def get_existing_urls(csv_path: Path) -> set:
    """Get all URLs already in CSV file."""
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path)
            return set(df['url'].dropna().tolist())
        except Exception:
            pass
    return set()


def combine_csvs(investing_csv: Path, serp_csv: Path, output_csv: Path):
    """
    Combine investing.com and SERP API CSV files:
    - Fix SERP API date formatting
    - Remove duplicates by URL
    - Drop rows without dates
    - Sort by date
    """
    dfs = []
    
    if investing_csv.exists():
        df_inv = pd.read_csv(investing_csv)
        dfs.append(df_inv)
        print(f"Loaded {len(df_inv)} articles from investing.com")
    
    if serp_csv.exists():
        df_serp = pd.read_csv(serp_csv)
        # Clean SERP API dates: extract date from strings like "4 days ago"
        df_serp['date'] = df_serp['date'].fillna('')
        dfs.append(df_serp)
        print(f"Loaded {len(df_serp)} articles from SERP API")
    
    if dfs:
        df_combined = pd.concat(dfs, ignore_index=True)
        
        # Remove exact duplicates by URL
        df_combined = df_combined.drop_duplicates(subset=['url'], keep='first')
        print(f"After deduplication: {len(df_combined)} unique articles")
        
        # Drop rows with missing dates
        initial_count = len(df_combined)
        df_combined = df_combined.dropna(subset=['date'])
        df_combined = df_combined[df_combined['date'].astype(str).str.strip() != '']
        dropped_count = initial_count - len(df_combined)
        
        if dropped_count > 0:
            print(f"Dropped {dropped_count} articles without dates")
        
        # Convert date column to datetime for proper sorting
        # Handle both ISO format and mixed formats
        df_combined['date'] = pd.to_datetime(df_combined['date'], errors='coerce')
        
        # Drop any rows where date conversion failed
        df_combined = df_combined.dropna(subset=['date'])
        
        # Sort by date (most recent first)
        df_combined = df_combined.sort_values('date', ascending=False).reset_index(drop=True)
        
        # Keep only wheat commodity
        df_combined = df_combined[df_combined['commodity'] == 'wheat']
        
        # Select standardized columns
        df_combined = df_combined[['commodity', 'title', 'date', 'source', 'description', 'url']]
        
        # Save as wheat_news.csv
        df_combined.to_csv(output_csv, index=False)
        print(f"\n✓ Combined: {len(df_combined)} wheat articles sorted by date → {output_csv}")
        return df_combined
    else:
        print("No CSV files found to combine.")
        return pd.DataFrame()


print('Helper functions defined.')


In [ ]:
# ── Main scraper loop ─────────────────────────────────────────────────────────

def scrape_commodity(config: dict, out_csv: Path):
    name      = config['name']
    base_url  = config['base_url']
    last_page = config['last_page']

    done_pages  = already_scraped_pages(out_csv)
    todo_pages  = [p for p in range(1, last_page + 1) if p not in done_pages]

    if not todo_pages:
        print(f'[{name}] All {last_page} pages already scraped. CSV: {out_csv}')
        return

    print(f'[{name}] Scraping {len(todo_pages)} pages (skipping {len(done_pages)} already done)…')
    scraper  = make_scraper()
    buffer   = []
    errors   = []

    for i, page in enumerate(tqdm(todo_pages, desc=name, unit='page'), start=1):
        url  = page_url(base_url, page)
        soup = fetch_page(scraper, url)

        if soup is None:
            errors.append(page)
            print(f'  [{name}] FAILED page {page} — skipping')
        else:
            records = parse_articles_investing(soup, name, page)
            buffer.extend(records)

        # Checkpoint: flush buffer to CSV every CHECKPOINT pages
        if i % CHECKPOINT == 0 and buffer:
            append_to_csv(buffer, out_csv)
            buffer.clear()
            tqdm.write(f'  [{name}] Checkpoint saved at page {page} → {out_csv}')

        # Polite delay (skip on last page)
        if i < len(todo_pages):
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    # Final flush
    if buffer:
        append_to_csv(buffer, out_csv)

    # Summary
    df = pd.read_csv(out_csv)
    print(f'\n[{name}] Done. {len(df)} articles → {out_csv}')
    if errors:
        print(f'  Failed pages (can re-run to retry): {errors}')
    return df

## Scrape Commodity News from Both Sources

In [ ]:
def scrape_all_sources():
    """
    Dual-source news scraper for wheat futures:
    1. Investing.com (direct web scraping)
    2. SERP API (financial news aggregator)
    
    All data is consolidated into a single wheat_news.csv file.
    """
    print("="*60)
    print("STARTING DUAL-SOURCE NEWS SCRAPING FOR WHEAT FUTURES")
    print("="*60 + "\n")

    # ── PHASE 1: Investing.com ──
    print("[PHASE 1/3] Scraping Investing.com...")
    for config in INVESTING_CONFIGS:
        scrape_commodity(config, OUT_CSV)

    # ── PHASE 2: SERP API ──
    print("\n[PHASE 2/3] Scraping SERP API financial news sources...\n")
    print(f"[SERP API] Fetching commodity news from {len(SERP_NEWS_SOURCES)} sources: {', '.join(SERP_NEWS_SOURCES)}\n")
    
    all_serp_records = []
    for i, config in enumerate(SERP_COMMODITIES, start=1):
        print(f"[{i}/{len(SERP_COMMODITIES)}] [SERP/{config['name'].upper()}] Fetching news...")
        
        records = fetch_commodity_news_serp(config['name'], config['query'])
        print(f"  [DEBUG] Fetched {len(records)} records from fetch_commodity_news_serp()")
        all_serp_records.extend(records)
        print(f"  [DEBUG] Total all_serp_records count: {len(all_serp_records)}")

    print(f"\n[DEBUG] Final all_serp_records length before append: {len(all_serp_records)}")
    if all_serp_records:
        print(f"  → {len(all_serp_records)} new articles to save\n")
        # Append SERP results directly to wheat_news.csv
        print(f"[DEBUG] Calling append_to_csv with {len(all_serp_records)} records")
        append_to_csv(all_serp_records, OUT_CSV)
        
        print("="*60)
        print(f"✓ SERP API: {len(all_serp_records)} total articles appended to {OUT_CSV.name}")
        print("="*60 + "\n")
    else:
        print("No articles fetched from SERP API.\n")

    # ── PHASE 3: Final Cleaning & Deduplication ──
    print("[PHASE 3/3] Cleaning, deduplicating, and finalizing dataset...")
    if OUT_CSV.exists():
        df_final = pd.read_csv(OUT_CSV)
        print(f"Initial count: {len(df_final)} articles from both sources")
        
        # Ensure all records are marked as 'wheat' commodity
        df_final['commodity'] = 'wheat'
        
        # Clean up dates
        df_final['date'] = df_final['date'].fillna('')
        
        # Remove exact duplicates by URL (keep first occurrence)
        initial_count = len(df_final)
        df_final = df_final.drop_duplicates(subset=['url'], keep='first')
        print(f"After deduplication: {len(df_final)} unique articles (removed {initial_count - len(df_final)} duplicates)")
        
        # Drop rows with missing or blank dates
        df_final = df_final[df_final['date'].astype(str).str.strip() != '']
        df_final['date'] = pd.to_datetime(df_final['date'], errors='coerce')
        df_final = df_final.dropna(subset=['date'])
        print(f"After date validation: {len(df_final)} articles with valid dates")
        
        # Sort by date (most recent first)
        df_final = df_final.sort_values('date', ascending=False).reset_index(drop=True)
        
        # Keep only required columns: commodity, title, date, source, description, url
        df_final = df_final[['commodity', 'title', 'date', 'source', 'description', 'url']]
        
        # Save the clean data back to wheat_news.csv
        df_final.to_csv(OUT_CSV, index=False)
        
        print(f"\n" + "="*60)
        print(f"✓ FINAL: {len(df_final)} unique wheat articles")
        print(f"✓ Saved to: {OUT_CSV.name}")
        print(f"✓ Date range: {df_final['date'].min().date()} to {df_final['date'].max().date()}")
        print(f"✓ Sources: {', '.join(df_final['source'].unique().tolist())}")
        print("="*60 + "\n")
        
        print("First 5 articles:")
        print(df_final.head())
        
        return df_final
    else:
        print(f"ERROR: No data found in {OUT_CSV.name}.")
        return pd.DataFrame()

In [15]:
# ── Execute the scraper ──
wheat_df = scrape_all_sources()
print("\n✓ Scraping complete! wheat_news.csv is ready for preprocessing.")

wheat:  20%|█▉        | 9/46 [00:42<02:39,  4.32s/page]

  [append_to_csv] Successfully appended 100 records to data/wheat_news.csv
  [wheat] Checkpoint saved at page 10 → data/wheat_news.csv


wheat:  41%|████▏     | 19/46 [01:25<01:56,  4.32s/page]

  [append_to_csv] Successfully appended 100 records to data/wheat_news.csv
  [wheat] Checkpoint saved at page 20 → data/wheat_news.csv


wheat:  63%|██████▎   | 29/46 [02:05<01:11,  4.19s/page]

  [append_to_csv] Successfully appended 100 records to data/wheat_news.csv
  [wheat] Checkpoint saved at page 30 → data/wheat_news.csv


wheat:  85%|████████▍ | 39/46 [02:47<00:31,  4.51s/page]

  [append_to_csv] Successfully appended 100 records to data/wheat_news.csv
  [wheat] Checkpoint saved at page 40 → data/wheat_news.csv


wheat: 100%|██████████| 46/46 [03:15<00:00,  4.25s/page]



  [append_to_csv] Successfully appended 60 records to data/wheat_news.csv

[wheat] Done. 1671 articles → data/wheat_news.csv

[PHASE 2/3] Scraping SERP API financial news sources...

[SERP API] Fetching commodity news from 4 sources: bloomberg.com, reuters.com, cnbc.com, investing.com

[1/1] [SERP/WHEAT] Fetching news...
✓ Fetched 6 articles from bloomberg.com (2013)
✓ Fetched 6 articles from bloomberg.com (2013)
✓ Fetched 2 articles from bloomberg.com (2014)
✓ Fetched 2 articles from bloomberg.com (2014)
✓ Fetched 10 articles from bloomberg.com (2015)
✓ Fetched 10 articles from bloomberg.com (2015)
✓ Fetched 10 articles from bloomberg.com (2016)
✓ Fetched 10 articles from bloomberg.com (2016)
✓ Fetched 10 articles from bloomberg.com (2017)
✓ Fetched 10 articles from bloomberg.com (2017)
✓ Fetched 10 articles from bloomberg.com (2018)
✓ Fetched 10 articles from bloomberg.com (2018)
✓ Fetched 10 articles from bloomberg.com (2019)
✓ Fetched 10 articles from bloomberg.com (2019)
✓ Fetched

In [16]:
# ── Test SERP API alone (without Investing.com scraping) ──
print("Testing SERP API scraper in isolation...")
print("This will skip Investing.com and test CNBC articles fetching only.\n")

test_records = fetch_commodity_news_serp('wheat', SERP_COMMODITIES[0]['query'])
print(f"\n✓ SERP API test returned {len(test_records)} total records")

if test_records:
    print("\nSample records from SERP API:")
    for i, record in enumerate(test_records[:3], start=1):
        print(f"  {i}. {record['source']} - {record['title'][:60]}...")
else:
    print("ERROR: No records returned from SERP API!")

Testing SERP API scraper in isolation...
This will skip Investing.com and test CNBC articles fetching only.

✓ Fetched 10 articles from cnbc.com (2023)
✓ Fetched 10 articles from cnbc.com (2023)
✓ Fetched 8 articles from cnbc.com (2026)
✓ Fetched 8 articles from cnbc.com (2026)
✓ Fetched 10 articles from investing.com (2011)
✓ Fetched 10 articles from investing.com (2012)
✓ Fetched 10 articles from investing.com (2011)
✓ Fetched 10 articles from investing.com (2012)
✓ Fetched 10 articles from investing.com (2013)
✓ Fetched 10 articles from investing.com (2014)
✓ Fetched 10 articles from investing.com (2013)
✓ Fetched 10 articles from investing.com (2014)
✓ Fetched 20 articles from investing.com (2015)
✓ Fetched 20 articles from investing.com (2015)
✓ Fetched 20 articles from investing.com (2016)
✓ Fetched 20 articles from investing.com (2016)
✓ Fetched 20 articles from investing.com (2017)
✓ Fetched 20 articles from investing.com (2017)
✓ Fetched 20 articles from investing.com (2018)
✓

In [17]:
# ── Save SERP records to CSV ──
print("\n" + "="*60)
print("SAVING SERP API RECORDS TO CSV")
print("="*60 + "\n")

# Append SERP results to wheat_news.csv (which already has Investing.com data)
append_to_csv(test_records, OUT_CSV)

print(f"✓ Appended {len(test_records)} SERP API records to wheat_news.csv")

# Now run cleanup/deduplication phase
print("\n[PHASE 3/3] Cleaning, deduplicating, and finalizing dataset...")
if OUT_CSV.exists():
    df_final = pd.read_csv(OUT_CSV)
    print(f"Initial count: {len(df_final)} articles from both sources")
    
    # Ensure all records are marked as 'wheat' commodity
    df_final['commodity'] = 'wheat'
    
    # Clean up dates
    df_final['date'] = df_final['date'].fillna('')
    
    # Remove exact duplicates by URL (keep first occurrence)
    initial_count = len(df_final)
    df_final = df_final.drop_duplicates(subset=['url'], keep='first')
    print(f"After deduplication: {len(df_final)} unique articles (removed {initial_count - len(df_final)} duplicates)")
    
    # Drop rows with missing or blank dates
    df_final = df_final[df_final['date'].astype(str).str.strip() != '']
    df_final['date'] = pd.to_datetime(df_final['date'], errors='coerce')
    df_final = df_final.dropna(subset=['date'])
    print(f"After date validation: {len(df_final)} articles with valid dates")
    
    # Sort by date (most recent first)
    df_final = df_final.sort_values('date', ascending=False).reset_index(drop=True)
    
    # Keep only required columns: commodity, title, date, source, description, url
    df_final = df_final[['commodity', 'title', 'date', 'source', 'description', 'url']]
    
    # Save the clean data back to wheat_news.csv
    df_final.to_csv(OUT_CSV, index=False)
    
    print(f"\n" + "="*60)
    print(f"✓ FINAL: {len(df_final)} unique wheat articles")
    print(f"✓ Saved to: {OUT_CSV.name}")
    print(f"✓ Date range: {df_final['date'].min().date()} to {df_final['date'].max().date()}")
    print(f"✓ Sources: {', '.join(df_final['source'].unique().tolist())}")
    print("="*60 + "\n")
    
    print("Article count by source:")
    print(df_final['source'].value_counts())
else:
    print(f"ERROR: No data found in {OUT_CSV.name}.")


SAVING SERP API RECORDS TO CSV

  [append_to_csv] Successfully appended 395 records to data/wheat_news.csv
✓ Appended 395 SERP API records to wheat_news.csv

[PHASE 3/3] Cleaning, deduplicating, and finalizing dataset...
Initial count: 1618 articles from both sources
After deduplication: 1601 unique articles (removed 17 duplicates)
After date validation: 1223 articles with valid dates

✓ FINAL: 1223 unique wheat articles
✓ Saved to: wheat_news.csv
✓ Date range: 2010-01-11 to 2026-05-04
✓ Sources: Investing.com, Reuters, Bloomberg, CNBC, Yolowire.com

Article count by source:
source
Reuters          472
Investing.com    364
Bloomberg        261
CNBC             124
Yolowire.com       2
Name: count, dtype: int64


In [18]:
# ── Debug: Check what happened to the SERP records ──
print("\n" + "="*60)
print("DEBUG: Investigating missing CNBC articles")
print("="*60 + "\n")

# Read the raw wheat_news.csv (before cleaning)
df_raw = pd.read_csv(OUT_CSV)
print(f"Total articles in CSV: {len(df_raw)}")

# Check CNBC articles specifically
cnbc_articles = df_raw[df_raw['source'] == 'cnbc.com']
print(f"CNBC articles found: {len(cnbc_articles)}")

if len(cnbc_articles) > 0:
    print("\nFirst 5 CNBC articles:")
    for idx, row in cnbc_articles.head().iterrows():
        print(f"  - Title: {row['title'][:50]}...")
        print(f"    Date: {row['date']}")
        print(f"    Date Type: {type(row['date'])}")
        print()
else:
    print("\nNo CNBC articles found. Checking all unique sources:")
    print(df_raw['source'].unique())
    
    # Check Bloomberg articles
    bloomberg = df_raw[df_raw['source'] == 'bloomberg.com']
    print(f"\nBloomberg articles: {len(bloomberg)}")
    if len(bloomberg) > 0:
        print("Sample Bloomberg article date:")
        print(f"  {bloomberg.iloc[0]['date']}")
        print(f"  Type: {type(bloomberg.iloc[0]['date'])}")


DEBUG: Investigating missing CNBC articles

Total articles in CSV: 1223
CNBC articles found: 0

No CNBC articles found. Checking all unique sources:
<StringArray>
['Investing.com', 'Reuters', 'Bloomberg', 'CNBC', 'Yolowire.com']
Length: 5, dtype: str

Bloomberg articles: 0


In [19]:
# ── Debug: Check date formats in test_records ──
print("\n" + "="*60)
print("DEBUG: Analyzing date formats in SERP records")
print("="*60 + "\n")

# Group test_records by date format
from collections import Counter

date_samples = {}
for record in test_records:
    date_str = str(record.get('date', ''))
    if date_str and date_str != '':
        if date_str not in date_samples:
            date_samples[date_str] = []
        if len(date_samples[date_str]) < 2:  # Keep 2 samples of each format
            date_samples[date_str].append(record)

print(f"Found {len(date_samples)} unique date formats in SERP records:\n")
for date_format, samples in list(date_samples.items())[:10]:
    print(f"Format: '{date_format}'")
    if samples:
        print(f"  Example title: {samples[0]['title'][:60]}...")
    print()


DEBUG: Analyzing date formats in SERP records

Found 360 unique date formats in SERP records:

Format: 'Jul 17, 2023'
  Example title: Wheat prices surge after Russia ends grain deal. And it's .....

Format: 'May 10, 2023'
  Example title: Geopolitical grains: Agricultural ETFs in a time of oversupp...

Format: 'Mar 31, 2023'
  Example title: Ukraine war live updates: Latest news on Russia and the ......

Format: 'Jul 20, 2023'
  Example title: India's rice export ban could send decade-high prices higher...

Format: 'Jan 13, 2023'
  Example title: Why inflation hit these 10 items hardest in 2022...

Format: 'Aug 21, 2023'
  Example title: Rice prices soar, fanning fears of food inflation spike in A...

Format: 'Jul 14, 2023'
  Example title: Russia-Ukraine war updates for July 14, 2023...

Format: 'Mar 2, 2023'
  Example title: Bangladesh: Firms should compensate poor nations ......

Format: 'Sep 25, 2023'
  Example title: Some soft commodity prices are surging, adding to ......

Form

In [20]:
# ── Analyze missing dates in SERP records ──
print("\n" + "="*60)
print("DEBUG: Missing dates analysis")
print("="*60 + "\n")

from collections import Counter

records_with_date = [r for r in test_records if r.get('date', '').strip()]
records_without_date = [r for r in test_records if not r.get('date', '').strip()]

print(f"Total SERP records: {len(test_records)}")
print(f"Records WITH date: {len(records_with_date)}")
print(f"Records WITHOUT date: {len(records_without_date)}")
print(f"Percentage with date: {len(records_with_date) / len(test_records) * 100:.1f}%")

# Check source distribution of records without dates
if records_without_date:
    print(f"\nSources of records WITHOUT dates:")
    sources_no_date = Counter(r['source'] for r in records_without_date)
    for source, count in sources_no_date.most_common():
        print(f"  {source}: {count}")
        
# Check if we can parse the dates that are present
print(f"\nTesting date parsing:")
dates_valid = 0
dates_invalid = 0
for record in records_with_date[:20]:
    try:
        pd.to_datetime(record['date'])
        dates_valid += 1
    except Exception as e:
        dates_invalid += 1
        print(f"  Failed to parse: '{record['date']}'")

if dates_invalid == 0:
    print(f"  ✓ All tested dates parsed successfully!")


DEBUG: Missing dates analysis

Total SERP records: 395
Records WITH date: 395
Records WITHOUT date: 0
Percentage with date: 100.0%

Testing date parsing:
  Failed to parse: '5 days ago'


In [21]:
# ── Solution: Properly rebuild wheat_news.csv with ALL sources ──
print("\n" + "="*60)
print("REBUILDING wheat_news.csv with ALL sources")
print("="*60 + "\n")

# Read the current Investing.com data
OUT_CSV_BACKUP = DATA_DIR / 'wheat_news_backup.csv'
df_investing = pd.read_csv(OUT_CSV)  # This still has Investing.com data
print(f"Current CSV has {len(df_investing)} articles from Investing.com")

# Create the combined dataset
all_records = []

# Add Investing.com records
for _, row in df_investing.iterrows():
    all_records.append(row.to_dict())

print(f"Added {len(all_records)} Investing.com records")

# Add SERP records with valid dates
serp_records_valid = [r for r in test_records if r.get('date', '').strip()]
all_records.extend(serp_records_valid)

print(f"Added {len(serp_records_valid)} SERP records with valid dates")
print(f"Total records before dedup: {len(all_records)}")

# Create DataFrame and deduplicate
df_combined = pd.DataFrame(all_records)

# Ensure proper columns
required_cols = ['commodity', 'title', 'date', 'source', 'description', 'url']
df_combined = df_combined[required_cols]

# Ensure all are 'wheat' commodity
df_combined['commodity'] = 'wheat'

# Remove duplicates by URL
initial_count = len(df_combined)
df_combined = df_combined.drop_duplicates(subset=['url'], keep='first')
print(f"After deduplication: {len(df_combined)} articles (removed {initial_count - len(df_combined)} duplicates)")

# Parse and sort dates
df_combined['date'] = pd.to_datetime(df_combined['date'], errors='coerce')
df_combined = df_combined.dropna(subset=['date'])
df_combined = df_combined.sort_values('date', ascending=False).reset_index(drop=True)

print(f"After date validation: {len(df_combined)} articles")

# Save to CSV
df_combined.to_csv(OUT_CSV, index=False)

print(f"\n" + "="*60)
print(f"✓ FINAL: {len(df_combined)} unique wheat articles")
print(f"✓ Saved to: {OUT_CSV.name}")
print(f"✓ Date range: {df_combined['date'].min().date()} to {df_combined['date'].max().date()}")
print("="*60 + "\n")

print("Article count by source:")
print(df_combined['source'].value_counts())
print("\n✓ Scraping complete! wheat_news.csv is ready for preprocessing.")


REBUILDING wheat_news.csv with ALL sources

Current CSV has 1223 articles from Investing.com
Added 1223 Investing.com records
Added 395 SERP records with valid dates
Total records before dedup: 1618
After deduplication: 1601 articles (removed 17 duplicates)
After date validation: 1223 articles

✓ FINAL: 1223 unique wheat articles
✓ Saved to: wheat_news.csv
✓ Date range: 2010-01-11 to 2026-05-04

Article count by source:
source
Reuters          472
Investing.com    364
Bloomberg        261
CNBC             124
Yolowire.com       2
Name: count, dtype: int64

✓ Scraping complete! wheat_news.csv is ready for preprocessing.


In [22]:
# ── DEBUG: Track which articles are being dropped ──
print("\n" + "="*60)
print("DEBUG: Tracking article filtering")
print("="*60 + "\n")

# Create DataFrame with all records
df_test = pd.DataFrame(all_records)
print(f"1. Starting with: {len(df_test)} articles")

# Check columns
required_cols = ['commodity', 'title', 'date', 'source', 'description', 'url']
print(f"   Columns available: {df_test.columns.tolist()}")

# Only keep required columns (and see if any are missing)
df_test = df_test[required_cols]
print(f"2. After selecting required columns: {len(df_test)} articles")

# Check for missing dates before converting
missing_date = df_test[df_test['date'].isna()]
print(f"3. Articles with NaN dates: {len(missing_date)}")

# Check for empty string dates  
empty_date = df_test[df_test['date'].astype(str).str.strip() == '']
print(f"4. Articles with empty string dates: {len(empty_date)}")

# Try to parse all dates and see which fail
print(f"5. Attempting to parse dates...")
df_test['date_parsed'] = pd.to_datetime(df_test['date'], errors='coerce')

# Check which dates failed to parse
failed_dates = df_test[df_test['date_parsed'].isna()]
print(f"   Failed to parse: {len(failed_dates)} articles")

if len(failed_dates) > 0:
    print(f"\n   Sample failed dates:")
    for idx, (orig, row) in enumerate(zip(failed_dates['date'].head(10), failed_dates[['date', 'source', 'title']].head(10).iterrows())):
        print(f"     '{orig}' from {row[1]['source']}")

# Final count after date validation
df_valid = df_test.dropna(subset=['date_parsed'])
print(f"\n6. After date validation: {len(df_valid)} articles")

# Check source distribution after filtering
print(f"\nSource distribution AFTER filtering:")
print(df_valid['source'].value_counts())


DEBUG: Tracking article filtering

1. Starting with: 1618 articles
   Columns available: ['commodity', 'title', 'date', 'source', 'description', 'url']
2. After selecting required columns: 1618 articles
3. Articles with NaN dates: 0
4. Articles with empty string dates: 0
5. Attempting to parse dates...
   Failed to parse: 395 articles

   Sample failed dates:
     'Jul 17, 2023' from cnbc.com
     'May 10, 2023' from cnbc.com
     'Mar 31, 2023' from cnbc.com
     'Jul 17, 2023' from cnbc.com
     'Jul 20, 2023' from cnbc.com
     'Jan 13, 2023' from cnbc.com
     'Aug 21, 2023' from cnbc.com
     'Jul 14, 2023' from cnbc.com
     'Mar 2, 2023' from cnbc.com
     'Sep 25, 2023' from cnbc.com

6. After date validation: 1223 articles

Source distribution AFTER filtering:
source
Reuters          472
Investing.com    364
Bloomberg        261
CNBC             124
Yolowire.com       2
Name: count, dtype: int64


In [23]:
# ── FIX: Use infer_datetime_format for better parsing ──
print("\n" + "="*60)
print("FIX: Rebuilding with better date parsing")
print("="*60 + "\n")

# Recreate the combined dataset
df_combined_fixed = pd.DataFrame(all_records)

# Select required columns
required_cols = ['commodity', 'title', 'date', 'source', 'description', 'url']
df_combined_fixed = df_combined_fixed[required_cols]

# Ensure all are 'wheat' commodity
df_combined_fixed['commodity'] = 'wheat'

# Remove duplicates by URL
initial_count = len(df_combined_fixed)
df_combined_fixed = df_combined_fixed.drop_duplicates(subset=['url'], keep='first')
print(f"After deduplication: {len(df_combined_fixed)} articles (removed {initial_count - len(df_combined_fixed)} duplicates)")

# Parse dates with infer_datetime_format=True for better handling
df_combined_fixed['date'] = pd.to_datetime(
    df_combined_fixed['date'], 
    errors='coerce',  # Convert unparseable to NaT
    format='mixed'    # Accept mixed date formats
)

# Check how many dates failed
failed = df_combined_fixed[df_combined_fixed['date'].isna()]
print(f"Failed to parse: {len(failed)} articles")

if len(failed) > 0:
    print("\nFailed dates by source:")
    print(failed['source'].value_counts())

# Drop articles without valid dates
df_combined_fixed = df_combined_fixed.dropna(subset=['date'])
df_combined_fixed = df_combined_fixed.sort_values('date', ascending=False).reset_index(drop=True)

print(f"\nAfter date validation: {len(df_combined_fixed)} articles")

# Save to CSV
df_combined_fixed.to_csv(OUT_CSV, index=False)

print(f"\n" + "="*60)
print(f"✓ FINAL: {len(df_combined_fixed)} unique wheat articles")
print(f"✓ Saved to: {OUT_CSV.name}")
print(f"✓ Date range: {df_combined_fixed['date'].min().date()} to {df_combined_fixed['date'].max().date()}")
print("="*60 + "\n")

print("Article count by source:")
source_counts = df_combined_fixed['source'].value_counts()
print(source_counts)
print(f"\nTotal sources: {len(source_counts)}")

print("\n✓ Scraping complete! wheat_news.csv is ready for preprocessing.")


FIX: Rebuilding with better date parsing

After deduplication: 1601 articles (removed 17 duplicates)
Failed to parse: 8 articles

Failed dates by source:
source
investing.com    7
cnbc.com         1
Name: count, dtype: int64

After date validation: 1593 articles

✓ FINAL: 1593 unique wheat articles
✓ Saved to: wheat_news.csv
✓ Date range: 2010-01-11 to 2026-05-04

Article count by source:
source
Reuters          472
investing.com    366
Investing.com    364
Bloomberg        261
CNBC             124
cnbc.com           4
Yolowire.com       2
Name: count, dtype: int64

Total sources: 7

✓ Scraping complete! wheat_news.csv is ready for preprocessing.


In [24]:
# ── CLEANUP: Normalize source names ──
print("\n" + "="*60)
print("CLEANUP: Normalizing source names")
print("="*60 + "\n")

# Reload the CSV
df_clean = pd.read_csv(OUT_CSV)

# Normalize source names (lowercase, remove .com)
def normalize_source(source):
    source_lower = source.lower()
    if 'bloomberg' in source_lower:
        return 'Bloomberg'
    elif 'reuters' in source_lower:
        return 'Reuters'
    elif 'cnbc' in source_lower:
        return 'CNBC'
    elif 'wsj' in source_lower or 'wall street' in source_lower:
        return 'WSJ'
    elif 'ft' in source_lower or 'financial times' in source_lower:
        return 'Financial Times'
    elif 'investing' in source_lower:
        return 'Investing.com'
    else:
        return source

df_clean['source'] = df_clean['source'].apply(normalize_source)

print("Source names BEFORE normalization were:")
print("  reuters.com, Investing.com, bloomberg.com, cnbc.com, Reuters, Bloomberg, Yolowire.com")

print("\nSource names AFTER normalization:")
print(df_clean['source'].value_counts())

# Save cleaned CSV
df_clean.to_csv(OUT_CSV, index=False)

print(f"\n✓ Normalized source names and saved to {OUT_CSV.name}")
print(f"✓ Total articles: {len(df_clean)}")
print("\n✓ wheat_news.csv is now ready for embedding extraction!")


CLEANUP: Normalizing source names

Source names BEFORE normalization were:
  reuters.com, Investing.com, bloomberg.com, cnbc.com, Reuters, Bloomberg, Yolowire.com

Source names AFTER normalization:
source
Investing.com    730
Reuters          472
Bloomberg        261
CNBC             128
Yolowire.com       2
Name: count, dtype: int64

✓ Normalized source names and saved to wheat_news.csv
✓ Total articles: 1593

✓ wheat_news.csv is now ready for embedding extraction!

✓ Normalized source names and saved to wheat_news.csv
✓ Total articles: 1593

✓ wheat_news.csv is now ready for embedding extraction!


In [25]:
wheat_df = pd.read_csv(OUT_CSV)
print("Size of final dataset:", len(wheat_df))

Size of final dataset: 1593


In [26]:
wheat_df

,commodity,title,date,source,description,url
0,wheat,CME Group reports April 2026 trading volume of...,2026-05-04 11:33:05,Investing.com,CME Group (CME) reported average daily volume ...,https://www.investing.com/news/assorted/cme-gr...
1,wheat,Manufacturing PMI and ISM data highlight Frida...,2026-04-30 18:01:03,Investing.com,As traders approach another pivotal day for fi...,https://www.investing.com/news/stock-market-ne...
2,wheat,"Chicago grains pare gains, mirroring crude oil",2026-04-30 14:44:37,Reuters,"PARIS/BEIJING, April 30 (Reuters) - Chicago gr...",https://www.investing.com/news/commodities-new...
3,wheat,Farm commodities surge to two-year high on Hor...,2026-04-29 11:56:49,Investing.com,Investing.com -- The extended closure of the S...,https://www.investing.com/news/economy-news/fa...
4,wheat,US wheat futures surge on strong EU barley exp...,2026-04-28 16:05:05,Investing.com,Investing.com -- US wheat futures climbed Thur...,https://www.investing.com/news/stock-market-ne...
...,...,...,...,...,...,...
1588,wheat,Memphis-Area Man Indicted Over Wheat Futures T...,2010-04-28 00:00:00,Bloomberg,U.S. And Israel Wage War Against Iran. Trump A...,https://www.bloomberg.com/news/articles/2010-0...
1589,wheat,Bigger U.S. corn acreage seen this year,2010-03-16 00:00:00,Reuters,Corn and soybean prices on the Chicago Board o...,https://www.reuters.com/article/business/bigge...
1590,wheat,"India's key measures, decisions on wheat",2010-03-03 00:00:00,Reuters,July 2009 - The government lifts a ban on whea...,https://www.reuters.com/article/business/india...
1591,wheat,Afghan war: Iconic images,2010-02-08 00:00:00,Reuters,... wheat to more than three-quarters of the w...,https://www.reuters.com/news/picture/afghan-wa...


In [27]:
"""
preprocess_news_embeddings.py
─────────────────────────────
Pipeline: wheat_news.csv  →  daily FinBERT [CLS] embeddings (768-d)
                           →  PCA reduction (768-d → 16-d)
                           +  daily sentiment scores (positive / negative / neutral)

Steps
  1. Load & clean text: concatenate title + description, strip journalist
     boilerplate from the description.
  2. Extract dense [CLS] embeddings via ProsusAI/finbert (NO sentiment head).
  3. Apply PCA dimensionality reduction: 768-d → 16-d.
  4. Extract sentiment scores via ProsusAI/finbert sentiment classification head.
  5. Aggregate to one (16,) tensor per calendar day (mean-pool multi-article
     days) and one mean sentiment score per day.
  6. Persist embeddings as dict[str, torch.Tensor] → data/wheat_news_embeddings.pt
     Persist sentiment as CSV → data/wheat_news_sentiment.csv

Usage
─────
    python preprocess_news_embeddings.py             # uses GPU if available
    python preprocess_news_embeddings.py --cpu       # force CPU
    python preprocess_news_embeddings.py --batch 16  # override batch size
"""

from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
from sklearn.decomposition import PCA

# ─── constants ──────────────────────────────────────────────────────────────
DATA_DIR   = "data"
INPUT_CSV  = "data/wheat_news.csv"
OUTPUT_PT  = "data/wheat_news_embeddings.pt"
OUTPUT_CSV = "data/wheat_news_sentiment.csv"

# FinBERT base model (NOT the sentiment-classification head)
MODEL_NAME = "ProsusAI/finbert"
MAX_LEN    = 512          # FinBERT max sequence length
BATCH_SIZE = 8            # default batch size; tune for your GPU RAM

# PCA dimensionality reduction
PCA_DIM    = 16           # reduce from 768-d to 16-d

# ─── regex: journalist boilerplate stripper ──────────────────────────────────
#
# Captures patterns like:
#   "By John Doe CHICAGO, March 27 (Reuters) - "
#   "By Julie Ingwersen MANHATTAN, Kansas, March 25 (Reuters) - "
#   "By Michael Hogan HAMBURG, March 19 (Reuters) - "
#   "* Wheat futures rise ..."  (Reuters bullet-point lead-ins)
#
# Strategy: match "By <words> <CITY>(, <State/Country>)? <rest> (<Source>) - "
# We also strip leading bullet-point markers ('*', '·', '•').
# ────────────────────────────────────────────────────────────────────────────
_BOILERPLATE_RE = re.compile(
    r"^"
    r"(?:\*\s*)?"                               # optional leading bullet
    r"(?:"
        r"By\s+[A-Z][a-zA-Z\s\-']+"            # "By <Reporter Name>"
        r"[A-Z]{2,}[\w\s,]*"                    # "CHICAGO, March 27" (CITY + date etc.)
        r"\([^)]+\)"                             # "(Reuters)" or "(Refinitiv)"
        r"\s*[-–—]\s*"                           # dash separator
    r")?"
    r"(?:\*\s*)?",                              # optional second bullet
    re.MULTILINE,
)


# ──────────────────────────────────────────────────────────────────────────────
#  1. TEXT CLEANING & CONCATENATION
# ──────────────────────────────────────────────────────────────────────────────

def clean_text(row: pd.Series) -> str:
    """
    Concatenate title + description into `clean_text`, stripping journalist
    boilerplate from the description prefix.

    Many descriptions are truncated with "..." — by prepending the title we
    recover more semantic signal even when the description is partial.
    """
    title       = str(row.get("title", "")).strip()
    description = str(row.get("description", "")).strip()

    # Strip leading boilerplate from description
    description = _BOILERPLATE_RE.sub("", description).strip()

    # Remove trailing ellipsis artefact left by scraper truncation
    if description.endswith("..."):
        description = description[:-3].strip()

    # Concatenate: "Title. Description"
    if description:
        return f"{title}. {description}"
    return title


# ──────────────────────────────────────────────────────────────────────────────
#  2. EMBEDDING EXTRACTION  ([CLS] hidden state, 768-d)
# ──────────────────────────────────────────────────────────────────────────────

def extract_cls_embeddings(
    texts: list[str],
    tokenizer: AutoTokenizer,
    model: AutoModel,
    device: torch.device,
    batch_size: int = BATCH_SIZE,
) -> torch.Tensor:
    """
    Pass *texts* through FinBERT and return the [CLS] token's last hidden
    state for every input.  Shape: (len(texts), 768).

    All inference runs inside `torch.no_grad()` to save memory.
    """
    all_embeddings: list[torch.Tensor] = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings"):
        batch_texts = texts[start : start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        # Move every tensor in the batch to the target device
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        # outputs.last_hidden_state → (batch, seq_len, 768)
        # [CLS] token is always at position 0
        cls_embeddings = outputs.last_hidden_state[:, 0, :]   # (batch, 768)

        # Move back to CPU immediately to free GPU memory
        all_embeddings.append(cls_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)           # (N, 768)


# ──────────────────────────────────────────────────────────────────────────────
#  2a. PCA DIMENSIONALITY REDUCTION  (768-d → 16-d)
# ──────────────────────────────────────────────────────────────────────────────

def apply_pca(embeddings: torch.Tensor, n_components: int = PCA_DIM) -> torch.Tensor:
    """
    Apply PCA to reduce embeddings from 768-d to n_components.
    
    Args:
        embeddings: Tensor of shape (N, 768)
        n_components: Target dimensionality (default 16)
    
    Returns:
        Reduced embeddings of shape (N, n_components)
    """
    print(f"[INFO] Applying PCA: {embeddings.shape[1]} → {n_components} dimensions")
    
    # Convert to numpy for scikit-learn
    embeddings_np = embeddings.numpy()
    
    # Fit and transform with PCA
    pca = PCA(n_components=n_components)
    reduced_embeddings_np = pca.fit_transform(embeddings_np)
    
    # Explained variance
    explained_var_ratio = pca.explained_variance_ratio_.sum()
    print(f"[INFO] Explained variance ratio: {explained_var_ratio:.4f}")
    
    # Convert back to torch
    reduced_embeddings = torch.from_numpy(reduced_embeddings_np).float()
    
    return reduced_embeddings


# ──────────────────────────────────────────────────────────────────────────────
#  2b. SENTIMENT SCORING  (FinBERT classification head → positive/negative/neutral)
# ──────────────────────────────────────────────────────────────────────────────

def extract_sentiment_scores(
    texts: list[str],
    tokenizer: AutoTokenizer,
    sentiment_model: AutoModelForSequenceClassification,
    device: torch.device,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    """
    Run FinBERT *with* the sentiment classification head to produce per-article
    softmax probabilities for [positive, negative, neutral] and a scalar
    sentiment score (positive - negative).

    Returns a DataFrame with columns:
        sentiment_pos, sentiment_neg, sentiment_neu, sentiment_score
    """
    all_scores: list[dict] = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Scoring sentiment"):
        batch_texts = texts[start : start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            logits = sentiment_model(**encoded).logits          # (batch, 3)
            probs = torch.softmax(logits, dim=-1).cpu()         # (batch, 3)

        # FinBERT label order: positive=0, negative=1, neutral=2
        for row in probs:
            pos, neg, neu = row[0].item(), row[1].item(), row[2].item()
            all_scores.append({
                "sentiment_pos":   pos,
                "sentiment_neg":   neg,
                "sentiment_neu":   neu,
                "sentiment_score": pos - neg,
            })

    return pd.DataFrame(all_scores)


# ──────────────────────────────────────────────────────────────────────────────
#  3. TEMPORAL ALIGNMENT  (daily mean-pooling)
# ──────────────────────────────────────────────────────────────────────────────

def aggregate_daily(
    df: pd.DataFrame,
    embeddings: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """
    Group embeddings by calendar date (YYYY-MM-DD).  For dates with multiple
    articles, return the element-wise mean.  Returns a dict mapping each
    date string to a (768,) tensor.
    """
    # Ensure a clean date column (string form, day-level)
    dates = df["date_day"].tolist()

    daily_map: dict[str, list[int]] = {}
    for idx, d in enumerate(dates):
        daily_map.setdefault(d, []).append(idx)

    result: dict[str, torch.Tensor] = {}
    for day, indices in sorted(daily_map.items()):
        stacked = embeddings[indices]                 # (k, 768)
        result[day] = torch.mean(stacked, dim=0)      # (768,)

    return result


# ──────────────────────────────────────────────────────────────────────────────
#  MAIN
# ──────────────────────────────────────────────────────────────────────────────

def main() -> None:
    parser = argparse.ArgumentParser(description="FinBERT news → daily embeddings")
    parser.add_argument("--cpu",   action="store_true",  help="Force CPU inference")
    parser.add_argument("--batch", type=int, default=BATCH_SIZE,
                        help=f"Batch size (default {BATCH_SIZE})")
    parser.add_argument("--input", type=str, default=str(INPUT_CSV),
                        help=f"Path to input CSV (default: {INPUT_CSV})")
    parser.add_argument("--output", type=str, default=str(OUTPUT_PT),
                        help=f"Path to output .pt file (default: {OUTPUT_PT})")
    args = parser.parse_args()

    # ── device ──────────────────────────────────────────────────────────────
    if args.cpu:
        device = torch.device("cpu")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")          # Apple Silicon GPU
    else:
        device = torch.device("cpu")

    print(f"[INFO] Using device: {device}")

    # ── 1. load & clean ────────────────────────────────────────────────────
    print(f"[INFO] Loading news from {args.input}")
    df = pd.read_csv(args.input)
    print(f"[INFO] Loaded {len(df)} articles")

    # Remove duplicates: same title AND source
    n_before = len(df)
    df = df.drop_duplicates(subset=["title", "source"], keep="first")
    n_after = len(df)
    n_dupes = n_before - n_after
    if n_dupes > 0:
        print(f"[INFO] Removed {n_dupes} duplicate(s) with same title & source")
    df = df.reset_index(drop=True)

    # Convert date column → datetime → calendar day string
    df["date"]     = pd.to_datetime(df["date"])
    df["date_day"] = df["date"].dt.strftime("%Y-%m-%d")

    # Build clean_text column
    df["clean_text"] = df.apply(clean_text, axis=1)

    # Sanity preview
    print(f"\n[PREVIEW] First cleaned article:\n  {df['clean_text'].iloc[0][:200]}…\n")

    texts = df["clean_text"].tolist()

    # ── 2. load FinBERT ────────────────────────────────────────────────────
    print(f"[INFO] Loading tokenizer & model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModel.from_pretrained(MODEL_NAME)
    model.to(device)
    model.eval()
    print(f"[INFO] Model loaded → {device}")

    # ── 3. extract embeddings ──────────────────────────────────────────────
    embeddings = extract_cls_embeddings(texts, tokenizer, model, device, args.batch)
    print(f"[INFO] Embedding matrix shape: {embeddings.shape}")   # (N, 768)

    # ── 3a. apply PCA ──────────────────────────────────────────────────────
    embeddings = apply_pca(embeddings, PCA_DIM)
    print(f"[INFO] Reduced embedding matrix shape: {embeddings.shape}")   # (N, 16)

    # ── 3b. extract sentiment scores ─────────────────────────────────────
    print(f"[INFO] Loading FinBERT sentiment classification head")
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    sentiment_model.to(device)
    sentiment_model.eval()

    sent_df = extract_sentiment_scores(texts, tokenizer, sentiment_model, device, args.batch)
    df = pd.concat([df.reset_index(drop=True), sent_df], axis=1)
    print(f"[INFO] Sentiment scores: mean={df['sentiment_score'].mean():.4f}, "
          f"std={df['sentiment_score'].std():.4f}")

    del sentiment_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ── 4. daily aggregation ───────────────────────────────────────────────
    daily_embeddings = aggregate_daily(df, embeddings)
    n_days    = len(daily_embeddings)
    n_articles = len(df)
    print(f"[INFO] {n_articles} articles aggregated into {n_days} daily vectors")

    # Daily sentiment: mean-pool per-article scores by day
    daily_sent = (df.groupby("date_day")[["sentiment_pos", "sentiment_neg",
                                          "sentiment_neu", "sentiment_score"]]
                    .mean()
                    .reset_index()
                    .rename(columns={"date_day": "date"}))
    daily_sent["article_count"] = (df.groupby("date_day").size()
                                     .values)
    daily_sent = daily_sent.sort_values("date").reset_index(drop=True)

    # ── 5. save ────────────────────────────────────────────────────────────
    output_path = Path(args.output)
    torch.save(daily_embeddings, output_path)
    print(f"[INFO] Saved daily embeddings → {output_path}")

    daily_sent.to_csv(OUTPUT_CSV, index=False)
    print(f"[INFO] Saved daily sentiment  → {OUTPUT_CSV}")

    # Quick verification
    sample_date = list(daily_embeddings.keys())[0]
    sample_vec  = daily_embeddings[sample_date]
    print(f"[CHECK] {sample_date} → tensor shape {sample_vec.shape}, "
          f"dtype {sample_vec.dtype}")

    # ── summary ────────────────────────────────────────────────────────────
    print("\n" + "═" * 60)
    print(f"  DONE — {n_days} daily embeddings ({PCA_DIM}-d after PCA)")
    print(f"       — {len(daily_sent)} daily sentiment scores")
    print(f"  Date range: {sorted(daily_embeddings.keys())[0]}"
          f" → {sorted(daily_embeddings.keys())[-1]}")
    print(f"  Outputs:    {output_path}")
    print(f"              {OUTPUT_CSV}")
    print("═" * 60)



main()


usage: ipykernel_launcher.py [-h] [--cpu] [--batch BATCH] [--input INPUT]
                             [--output OUTPUT]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/isha/Library/Jupyter/runtime/kernel-v30e0f192f7a070552fe225fcb778c9e6ed4d342b6.json


SystemExit: 2

/Users/isha/Desktop/Projects/DeepTemporalModelsWithAttentionAndSkipConnectionsForWheatFutures/DeepTemporalModelsWithAttentionAndSkipConnectionsForWheatFutures_venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
